# Inter-USV Interval Mixture Models — Fits and Plots

This notebook fits and visualises mixture models on the distribution of **inter-USV intervals** (in seconds, log-transformed) for one or more sessions. It produces:

1. A **master DataFrame** of every interval across the chosen session list.
2. An **information-criterion sweep** over candidate component counts for both Gaussian and Student-t mixtures.
3. A **bootstrap likelihood-ratio test** that selects the best `K` per family.
4. The **fit plots**: log-interval histogram with overlay, Q-Q diagnostic, and a table of fitted component parameters per sex.

Compute is split from plotting so you can re-render figures without refitting.


In [ ]:
### Imports and color settings

# Auto-reload package modules whenever they change on disk, so edits to
# the usv_playpen package are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import polars as pls

from usv_playpen.os_utils import configure_path
import usv_playpen.visualizations.usv_interval_summary_statistics as ivs

base_path = Path.cwd().parent
fm.fontManager.addfont(base_path / "fonts/Helvetica.ttf")
plt.style.use(base_path / "_config/usv_playpen.mplstyle")

with open(base_path / "_parameter_settings" / "visualizations_settings.json", 'r') as vis_settings_file:
    vis_settings = json.load(vis_settings_file)

with open(base_path / "_parameter_settings" / "analyses_settings.json", 'r') as ana_settings_file:
    ana_settings = json.load(ana_settings_file)

male_color = vis_settings["male_colors"][0]
female_color = vis_settings["female_colors"][0]
line_color = "#202020"

# All compute / display knobs live in analyses_settings.json -> compute_inter_usv_interval_distributions
usv_interval_cfg = ana_settings["compute_inter_usv_interval_distributions"]

## Configuration

Set the session list(s) to include, the candidate component counts, and the bootstrap size. The bootstrap step is the only slow part of the notebook (cell **[5]**).


In [ ]:
### Configuration

# Single source of truth for paths and labels. All numeric parameters live
# in analyses_settings.json; this cell only assigns notebook-local convenience
# aliases.
output_directory = usv_interval_cfg["output_directory"]
session_lists = [str(Path(configure_path(p))) for p in usv_interval_cfg["session_lists"]]

interval_types = ("s2s", "e2s")
mode_label = {
    "s2s": "start-to-start USV intervals",
    "e2s": "end-to-start USV intervals",
}

# Plot-only knobs (read straight from JSON; not archived in the HDF5)
plot_log_xlims = tuple(usv_interval_cfg["plot_log_xlims"])
bins_per_sex = usv_interval_cfg["bins_per_sex"]
tau = usv_interval_cfg["tau"]
model_class = usv_interval_cfg["model_class"]

print(f"output_directory: {output_directory}")
print(f"session_lists ({len(session_lists)}):")
for p in session_lists:
    print(f"  {p}")
print(f"model_class: {model_class!r}")

## Compute

The cells in this section compute everything **in memory** and persist a single
self-describing HDF5 archive `usv_interval_analysis_<YYYYMMDD>_<HHMMSS>.h5` to
`output_directory`. **Run them once; after that the plot cells (below) read
from the most-recent archive on disk, so you can iterate on plot styling
across kernel restarts.**

The archive layout (see `usv_playpen.analyses.usv_interval_archive`):

- `/attrs` -- every JSON parameter that drove the run, plus `created_at_iso`,
  `git_sha`, `source_lists`, `n_sessions_loaded`.
- `/<mode>/intervals` -- tidy one-row-per-inter-USV interval table.
- `/<mode>/drop_counts` -- per-sex count of dropped non-positive intervals.
- `/<mode>/gmm_fits` -- IC sweep with all four ICs and per-component params
  (`logmean_k`, `logsd_k`, `weight_k`, `nu_k`) per `(sex, n_comp, rep)` row.
- `/<mode>/bootstrap_lrt` -- per-pair LRT summary plus `K_selected_step_up`.
- `/<mode>/bootstrap_lrt_null` -- long-form bootstrap LR null distribution.
- `/<mode>/attrs` -- `alpha_effective`, `K_selected_male`, `K_selected_female`.

## Compute step 1 — Build the master DataFrame

Walks every session in the configured list, reads its `*_usv_summary.csv`, computes consecutive inter-USV intervals in seconds, and appends them with sex / cohort / estrous metadata to a single Polars DataFrame. Drop counts (sessions skipped because of missing or malformed CSVs) are reported.


In [ ]:
### [Compute] Build master inter-USV interval DataFrame and drop counts

# Reads session lists, computes per-session same-emitter inter-USV intervals in both modes.
# Pure compute -- no disk side effects until the "Save archive" cell.
usv_interval_df, usv_interval_summary = ivs.build_master_usv_interval_dataframe(
    session_lists=session_lists,
    noise_col_id=usv_interval_cfg["noise_col_id"],
    noise_categories=usv_interval_cfg["noise_categories"],
)

if usv_interval_df.height == 0:
    raise RuntimeError(
        "Master inter-USV interval DataFrame is empty. Check session_lists in analyses_settings.json: "
        "each line must resolve to a session containing both a "
        "'*_points3d_translated_rotated_metric.h5' tracking file and a "
        "'*_usv_summary.csv' USV file."
    )

print(f"\nLoaded inter-USV intervals from {usv_interval_summary['n_sessions_loaded']} session(s).")
for it in interval_types:
    sub = usv_interval_df.filter(pls.col("interval_type") == it)
    n_male   = sub.filter(pls.col("sex") == "male").height
    n_female = sub.filter(pls.col("sex") == "female").height
    drops    = usv_interval_summary["n_dropped"][it]
    print(f"  [{it}] male = {n_male}, female = {n_female}  "
          f"(dropped non-positive: male = {drops['male']}, female = {drops['female']})")

## Compute step 2 — Information-criterion sweep

Fit both a `GaussianMixture` and a Student-t mixture for `K = 1, 2, … K_max` components. Report `BIC` and `AIC` for each `K`. The minimum-IC point gives the preliminary model order; the bootstrap LRT below confirms (or rejects) it.


In [ ]:
### [Compute] GMM / t-mixture IC sweep

# Returns the full sweep (all four ICs + per-component parameters) per mode.
gmm_fits_by_mode = {}
for it in interval_types:
    sub = usv_interval_df.filter(pls.col("interval_type") == it)
    df_results = ivs.run_bic_sweep(
        usv_interval_df=sub,
        n_components_min=usv_interval_cfg["n_components_min"],
        n_components_max=usv_interval_cfg["n_components_max"],
        n_repeats=usv_interval_cfg["n_repeats"],
        max_modes_reported=usv_interval_cfg["max_modes_reported"],
        random_seed_base=usv_interval_cfg["random_seed_base"],
        model_class=model_class,
    )
    gmm_fits_by_mode[it] = df_results
    print(f"[{mode_label[it]}] sweep produced {df_results.height} rows.")

## Compute step 3 — Bootstrap likelihood-ratio test

For each candidate `K` we resample the data, refit the model, and compare the log-likelihood of `K` vs. `K-1` components. The empirical null distribution of the log-likelihood-ratio gives a p-value. **This is the slow cell**, scaling roughly linearly with bootstrap count and `K_max`.


In [ ]:
### [Compute] Bootstrap LRT (this is the slow cell)

# Pure compute helper: returns the in-memory sweep dict per mode. The
# step-up rule is applied later, when the archive is written, so the
# alpha / bonferroni settings live in a single place (the JSON).
lrt_sweep_by_mode = {}
for it in interval_types:
    sub = usv_interval_df.filter(pls.col("interval_type") == it)
    male_arr   = sub.filter(pls.col("sex") == "male")["interval_s"].to_numpy()
    female_arr = sub.filter(pls.col("sex") == "female")["interval_s"].to_numpy()
    print(f"[{mode_label[it]}] running bootstrap LRT "
          f"(B={usv_interval_cfg['bootstrap_lrt_B']}, N_sub={usv_interval_cfg['bootstrap_lrt_n_subsample']}, "
          f"model={model_class})...")
    lrt_sweep_by_mode[it] = ivs.run_bootstrap_lrt_sweep(
        intervals_by_key={"male": male_arr, "female": female_arr},
        n_components_min=usv_interval_cfg["n_components_min"],
        n_components_max=usv_interval_cfg["n_components_max"],
        B=usv_interval_cfg["bootstrap_lrt_B"],
        n_subsample=usv_interval_cfg["bootstrap_lrt_n_subsample"],
        model_class=model_class,
        n_init_obs=usv_interval_cfg["gmm_n_init"],
        n_init_boot=max(1, usv_interval_cfg["gmm_n_init"] - 7),
        reg_covar=usv_interval_cfg["gmm_reg_covar"],
        seed=usv_interval_cfg["random_seed_base"],
    )

## Compute step 4 — Persist the fits

Bundle the master DataFrame, the IC sweep, the LRT results, and the best-fit models into a single HDF5 archive. Plotting cells below read from this archive, so you can re-plot without rerunning the (slow) compute steps.


In [ ]:
### [Compute] Save archive (single HDF5 with both modes)

# Writes usv_interval_analysis_<YYYYMMDD>_<HHMMSS>.h5 in output_directory. The
# step-up LRT selection is applied here so the per-mode K_selected_* attrs
# in the archive match the alpha / bonferroni settings in analyses_settings.json.
h5_path = ivs.save_notebook_archive_to_h5(
    output_directory=output_directory,
    usv_interval_df=usv_interval_df,
    usv_interval_summary=usv_interval_summary,
    usv_interval_cfg=usv_interval_cfg,
    gmm_fits_by_mode=gmm_fits_by_mode,
    lrt_sweep_by_mode=lrt_sweep_by_mode,
)
print(f"archive written: {h5_path}")

## Plot

These cells **read everything from the HDF5 archive** in `output_directory`,
so you can iterate on plot styling without re-running anything in the Compute
section -- even across kernel restarts. By default we pick the most-recent
archive (`ivs.find_latest_archive`); set `h5_path` manually to re-render an
older run.

## Plot step 1 — Output controls

Choose the archive to plot from and where to write figures. Switching archives lets you produce comparison figures across cohorts without re-fitting.


In [ ]:
### Figure-saving controls + select archive to read from

figure_save_dir = Path(configure_path(usv_interval_cfg["figures_directory"])) if usv_interval_cfg["figures_directory"] else None
if figure_save_dir is not None:
    figure_save_dir.mkdir(parents=True, exist_ok=True)

save_fig_bool = False
save_fig_format = "svg"

# Pick up the most-recent archive so plot cells survive kernel restarts.
# Override by assigning `h5_path = Path("...absolute/path/usv_interval_analysis_xxx.h5")`.
h5_path = ivs.find_latest_archive(output_directory)
print(f"reading from: {h5_path}")

## Plot step 2 — log-interval histogram (sanity check)

First diagnostic: a histogram of `log(interval)` per sex with the empirical kernel density. Confirms that the data is bimodal-ish (short intra-bout intervals + long inter-bout intervals) before any model is fit.


In [ ]:
### [Plot] log_interval distribution (sanity check)

for it in interval_types:
    sub = ivs.load_intervals_from_h5(str(h5_path), it)
    fig, ax, hist_stats = ivs.plot_log_usv_interval_histograms(
        usv_interval_df=sub,
        bins=max(bins_per_sex.values()),
        male_color=male_color,
        female_color=female_color,
        xlims=plot_log_xlims,
    )
    ax.set_title(f"log_interval distribution -- {mode_label[it]}")
    if save_fig_bool and figure_save_dir is not None:
        fig.savefig(figure_save_dir / f"ivi_log_histogram_{it}.{save_fig_format}", bbox_inches="tight")
    plt.show()
    display({**hist_stats, "interval_type": it})

## Plot step 3 — Bootstrap LRT null distribution

Visualises the bootstrapped LRT distribution under the null and the observed test statistic for each `K`-vs-`(K-1)` comparison. The p-value is the right-tail mass.


In [ ]:
### [Plot] Bootstrap LRT null-distribution panel

for it in interval_types:
    sweep = ivs.load_lrt_sweep_from_h5(str(h5_path), it)
    fig, _ = ivs.plot_bootstrap_lrt_panel(sweep)
    fig.suptitle(f"Bootstrap LRT null distributions -- {mode_label[it]}", y=1.02)
    fig.tight_layout()
    if save_fig_bool and figure_save_dir is not None:
        fig.savefig(figure_save_dir / f"ivi_bootstrap_lrt_{it}.{save_fig_format}", bbox_inches="tight")
    plt.show()
    selected = ivs.selected_K_from_h5(str(h5_path), it)
    print(f"[{mode_label[it]}] step-up LRT selected K: {selected}")

## Plot step 4 — IC curves

BIC and AIC vs. `K`. The elbow / minimum is the IC-preferred model order, which can differ from the LRT-preferred order — both numbers are reported in the title.


In [ ]:
### [Plot] BIC and AIC vs n_components

# Twin axes (male = left, female = right). Same-size circles along each curve;
# outlined black square at the K selected by the bootstrap-LRT step-up rule.
for it in interval_types:
    df_ic = ivs.load_gmm_fits_from_h5(str(h5_path), it)
    selected = ivs.selected_K_from_h5(str(h5_path), it)
    for ic_to_plot in ("bic", "aic"):
        fig, (ax_left, ax_right), _ = ivs.plot_ic_curves(
            df_results=df_ic,
            male_color=male_color,
            female_color=female_color,
            ic_col=ic_to_plot,
            selected_n_components=selected,
        )
        ax_left.set_title(f"{ic_to_plot.upper()} vs n_components -- {mode_label[it]}")
        if save_fig_bool and figure_save_dir is not None:
            fig.savefig(figure_save_dir / f"ivi_{ic_to_plot}_curve_{it}.{save_fig_format}", bbox_inches="tight")
        plt.show()

## Plot step 5 — Best-fit overlay

Histogram of `log(interval)` with the LRT-selected mixture density overlaid; an inset Q-Q plot diagnostic compares empirical to model quantiles. Deviations in the tails are usually where the Gaussian assumption breaks down — the t-mixture variant exists precisely for that case.


In [ ]:
### [Plot] Best-fit overlay (with Q-Q inset) at the LRT-selected K

# Reconstruct the best-rep fitted mixture for (sex, K=K_selected) directly
# from the archived sweep -- no refit, no EM. Component parameters
# (logmean_k, logsd_k, weight_k, nu_k) are read straight from the row, so
# the rendered plot matches the archive bit-for-bit even months later.
# Triangles mark each component log-mean (apex on the curve); a bold
# left-aligned text legend in the upper-right maps (a)/(b)/... to
# medians in seconds. The Q-Q inset is positioned automatically below
# the legend for males (so it tracks the legend height regardless of K)
# and explicitly in the upper-left, smaller, for females.
color_for = {"male": male_color, "female": female_color}
model_class_label = {"t": "t-distribution", "gauss": "Gaussian"}.get(model_class, model_class)

# Toggle: when True, overlay each individual mixture component pdf
# on the histogram (each with a distinct colour + linestyle).
show_mixture_components = False

for it in interval_types:
    sub = ivs.load_intervals_from_h5(str(h5_path), it)
    selected = ivs.selected_K_from_h5(str(h5_path), it)
    intervals_by_sex = {
        "male":   sub.filter(pls.col("sex") == "male")["interval_s"].to_numpy(),
        "female": sub.filter(pls.col("sex") == "female")["interval_s"].to_numpy(),
    }
    for sex, intervals_sec in intervals_by_sex.items():
        if intervals_sec.size < 2 or sex not in selected:
            continue
        n_comp = int(selected[sex])
        gmm, gmm_order = ivs.load_best_fit_from_h5(
            h5_path=str(h5_path),
            interval_type=it,
            sex=sex,
            K=n_comp,
        )

        if sex == "male":
            kw = dict(auto_inset_below_legend=True, auto_inset_size=(0.45, 0.45))
        else:
            kw = dict(auto_inset_below_legend=True, auto_inset_size=(0.25, 0.25))

        fig_fit, ax_fit, fit_summary = ivs.plot_best_fit_with_annotations(
            intervals_sec=intervals_sec,
            gmm=gmm,
            gmm_order=gmm_order,
            color=color_for[sex],
            figsize=(5, 5),
            bins=bins_per_sex[sex],
            xlims=plot_log_xlims,
            tau=tau,
            legend_corner="upper right",
            show_components=show_mixture_components,
            **kw,
        )
        ax_fit.set_title(
            f"{mode_label[it].capitalize()} {model_class_label} "
            f"mixture model LRT-selected K={n_comp}"
        )
        if save_fig_bool and figure_save_dir is not None:
            fig_fit.savefig(
                figure_save_dir / f"ivi_best_fit_{model_class}_{sex}_{it}.{save_fig_format}",
                bbox_inches="tight",
            )
        plt.show()
        print(f"  [{sex}] log-log Pearson r (Q-Q) = {fit_summary['qq_pearson_r']:.4f}")

## Plot step 6 — Print fitted parameters

Final summary: per-sex table of fitted component parameters (log-mean and log-sd, or log-scale and df for the t-mixture) at the LRT-selected `K`. Useful for downstream comparisons across cohorts and to cite as numerical results in a paper.


In [ ]:
### [Print] Fitted component parameters (log-mean / log-sd per sex)

# Pulls the LRT-selected K best-rep components straight out of the
# archive and prints them in JSON form so the values are easy to paste
# into reports or downstream config. Components are sorted ascending by
# log-mean (matching the (a)/(b)/(c) triangle labels in cell 12); inner
# lists are formatted on a single line each.
import json
import numpy as np

for it in interval_types:
    try:
        selected = ivs.selected_K_from_h5(str(h5_path), it)
    except ValueError:
        continue
    if not selected:
        continue

    sex_blocks = []
    for sex in ("male", "female"):
        if sex not in selected:
            continue
        K = int(selected[sex])
        gmm, _ = ivs.load_best_fit_from_h5(
            h5_path=str(h5_path), interval_type=it, sex=sex, K=K,
        )
        means = [round(float(m), 5) for m in np.asarray(gmm.means_).flatten()]
        sds = [round(float(s), 5) for s in np.sqrt(np.asarray(gmm.covariances_).reshape(-1))]
        sex_blocks.append(
            f'    "{sex}": {{\n'
            f'      "means": {json.dumps(means)},\n'
            f'      "sds": {json.dumps(sds)}\n'
            f'    }}'
        )

    print(f"[{it}]")
    print("{")
    print('  "gmm_params": {')
    print(",\n".join(sex_blocks))
    print("  }")
    print("}")
    print()
